In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

cfg = common.load_config("dev")

def count(stage):
    try:
        return (spark.read.format("binaryFile").option("recursiveFileLookup", "true")
                .load(f"/Volumes/{cfg['catalog']}/bronze/{stage}/").select("path").count())
    except Exception as e:
        if "PATH_NOT_FOUND" in str(e):
            return 0
        raise

got = {stage: count(stage) for stage in ("landing", "processed", "rejected")}
want = {"landing": 0, "processed": {"bhg": 209, "fqf": 178}[cfg["code"]], "rejected": 0}
print(got, "PASS" if got == want else "FAIL")